In [ ]:

MODEL_NAME ="meta-llama/Llama-2-7b-chat-hf"    #"openai-community/gpt2"

import os
import gc
import re
import json
import random
import string
import torch
import numpy as np
#import evaluate
import random
from tqdm import tqdm
from functools import partial
from tabulate import tabulate
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import huggingface_hub
import torch.nn.functional as F
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from huggingface_hub import login
import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import FactoredMatrix, HookedTransformer
from transformer_lens.model_bridge import TransformerBridge
from datasets import load_dataset

login(token="enter_token_from_bucket")

torch.set_grad_enabled(True)                                            
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = TransformerBridge.boot_transformers(
    MODEL_NAME,
    dtype=torch.float16,
    n_devices=2,)


In [ ]:



#Checks whether the generated text contains any acceptable answer, first by lowercase match and then by full normalization.
def is_correct(text, options):

    def _lowercase_only(s):
        return s.lower()
    
    
    def _normalize(s):
        s = s.lower()
        punctuation_to_strip = string.punctuation.replace(".", "").replace("-", "")
        s = s.translate(str.maketrans("", "", punctuation_to_strip))
        s = re.sub(r'(?<!\d)\.(?!\d)', '', s)   # drop '.' unless part of a number
        s = re.sub(r'(?<!\d)-(?!\d)', ' ', s)   # drop '-' unless part of a number
        s = re.sub(r"\s+", " ", s).strip()
        return s
    def contains_phrase(haystack, phrase) :
        
        if not phrase:
            return False
        pattern = r"\b" + re.escape(phrase) + r"\b"
        return bool(re.search(pattern, haystack))


    if not text or not options:
        return False

    text_lower = _lowercase_only(text)
    if any(contains_phrase(text_lower, _lowercase_only(opt)) for opt in options):
        return True

    text_norm = _normalize(text)
    if any(contains_phrase(text_norm, _normalize(opt)) for opt in options):
        return True

    return False


#Builds a prompt from the substituted context and the question.
def build_conflict_prompt(row):

    context = row["sub_context"]
    question = row["question"]
    prompt = f"{context}\n\nQuestion: {question}\nAnswer in few words:"

    return prompt


In [ ]:
#Generates an answer for each example and sorts it into the memory-answered or context-answered list, saving both to JSON.
def answer_from_mem_or_context(dataset_arg):
  list_of_memory_answers_data = []
  list_of_context_answers_data = []

  data_QA=dataset_arg
  count=0
  for dicti in data_QA:
      print("in i=",count)
      count+=1
      prompt = build_conflict_prompt(dicti)
      input_ids = model . to_tokens ( prompt , prepend_bos = True )


      full_tokens = model.generate(
          input_ids,
          max_new_tokens=15,
          do_sample=False  # Set to True for temperature/top-p sampling
      )

      output=model.to_string(full_tokens)[0]
      #print("output",output)
      bos_str = model.to_string([model.tokenizer.bos_token_id])
      bos_len = len(bos_str)
      generated_only = output[len(prompt)+ bos_len:]
      print(generated_only)
      if is_correct(generated_only,dicti["org_answer"]):
          list_of_memory_answers_data.append(dicti)
      if is_correct(generated_only,dicti["sub_answer"]):
          list_of_context_answers_data.append(dicti)

      with open(file_path_memory_answers, "w") as f:
          json.dump(list_of_memory_answers_data, f)

      with open(file_path_context_answers, "w") as f:
          json.dump(list_of_context_answers_data, f)

      gc.collect()
      torch.cuda.empty_cache()

      return list_of_memory_answers_data,list_of_context_answers_data

In [ ]:
#Groups examples by question, removes questions that appear in both lists, and splits each list 80/20 into train and test.
def filter_Q_and_split_train_test(list_of_memory_answers_data,list_of_context_answers_data):
  #ORDERING THE DATA OF FROM CONTEXT OR FROM MEMORY BY QOUSTION AND FINALY SPLITING IT TO TRAIN AND TEST FOR EACH KIND
  mem_dict_by_Q={}
  for data_item in list_of_memory_answers_data:
      Q_key=data_item['question']
      if Q_key not in mem_dict_by_Q:
          mem_dict_by_Q[Q_key] = [data_item]

      else:
          mem_dict_by_Q[Q_key].append(data_item)


  context_dict_by_Q={}
  for data_item in list_of_context_answers_data:
      Q_key=data_item['question']
      if Q_key not in context_dict_by_Q:
          context_dict_by_Q[Q_key] = [data_item]

      else:
          context_dict_by_Q[Q_key].append(data_item)


  shared_keys = mem_dict_by_Q.keys() & context_dict_by_Q.keys()
  '''print("shared Qustions between datasets?:",bool(shared_keys))
  print(len(shared_keys))
  print("number of diffrent Q in list_of_memory_answers_data=",len(mem_dict_by_Q.keys()))
  print("number of diffrent Q in list_of_context_answers_data=",len(context_dict_by_Q.keys()))'''

  from_mem_close_call = {k: v for k, v in mem_dict_by_Q.items() if k in shared_keys}
  from_context_close_call = {k: v for k, v in context_dict_by_Q.items() if k in shared_keys}

  for k in shared_keys:
      del mem_dict_by_Q[k]
      del context_dict_by_Q[k]

  def split_train_from_dict(dictt):
      keys = list(dictt.keys())
      split = int(0.8 * len(keys))
      train_keys = keys[:split]
      test_keys = keys[split:]

      train = {k: dictt[k] for k in train_keys}
      test = {k: dictt[k] for k in test_keys}
      return [x for v in train.values() for x in v],[x for v in test.values() for x in v]


  context_only_train,context_only_test=split_train_from_dict(context_dict_by_Q)

  rng = random.Random(42)
  rng.shuffle(context_only_train)
  rng.shuffle(context_only_test)

  context_only_train=context_only_train[:int(0.8*200)]
  context_only_test=context_only_test[:int(0.2*200)]
  mem_only_train,mem_only_test=split_train_from_dict(mem_dict_by_Q)
  return context_only_train,context_only_test,mem_only_train,mem_only_test

In [ ]:

#Computes a per-head importance score for one prompt and target token, using a probability-sum variant of the score.
def importance_for_example_prob_sum(prompt, target_token_id) :
    tokens = model.to_tokens(prompt)

    model.reset_hooks(including_permanent=True)
    model.set_use_attn_result(True)
    try:
        logits, cache = model.run_with_cache(
            tokens,
            return_type="logits",
            names_filter=lambda name: name.endswith("hook_result"),
        )
    except Exception as e:
        model.reset_hooks(including_permanent=True)
        model.zero_grad(set_to_none=True)
        gc.collect()
        torch.cuda.empty_cache()
        return None

    importance = torch.zeros(n_layers, n_heads)
    for l in range(n_layers):
        result = cache[utils . get_act_name ( "result" , l )][0,-1,:,:]       
        logit_lens_logits = model.unembed(model.ln_final(result))

        all_probs = torch.softmax(logit_lens_logits, dim=-1)[:,target_token_id]

        importance[l] = all_probs


    model.reset_hooks(including_permanent=True)
    model.zero_grad(set_to_none=True)
    del logits ,cache
    gc.collect()
    torch.cuda.empty_cache()

    return importance

#Averages per-head importance for one answer type across a dataset.
def compute_dataset_importance(dataset,key_ans,prompt_func,importance_for_example):
    count=0
    total = torch.zeros(n_layers, n_heads)
    for i,Q_A in enumerate(dataset):
        #print("in i=",i)
        prompt=prompt_func(Q_A)

        target_id= model.to_tokens( Q_A[key_ans][0] , prepend_bos = False )[0,0].item()
        #print(model . to_string ( target_id ))
        I=importance_for_example(prompt, target_id)
        if I != None:
            count += 1
            total += I
    print("sucsees in",count)
    return total / count


#Computes per-head importance for two answer types over a dataset, and their difference (memory vs. context preference).
def compute_dataset_importance_diff(dataset,key_ans1,key_ans2,prompt_func,importance_for_example):
    count=0
    total_1=torch.zeros(n_layers, n_heads)
    total_2=torch.zeros(n_layers, n_heads)
    total_diff = torch.zeros(n_layers, n_heads)
    total_multi = torch.zeros(n_layers, n_heads)
    for Q_A in tqdm(dataset):
        #print("in i=",i)
        prompt=prompt_func(Q_A)
        target_id1= model.to_tokens( Q_A[key_ans1][0] , prepend_bos = False )[0,0].item()
        target_id2= model.to_tokens( Q_A[key_ans2][0] , prepend_bos = False )[0,0].item()
        #print(model . to_string ( target_id ))
        I1=importance_for_example(prompt, target_id1)
        I2 =importance_for_example(prompt, target_id2)

        if I1 != None and I2 != None:
            count += 1
            total_1+=I1
            total_2+=I2
            total_diff += (I1-I2)
            total_multi+=(I1*I2)
    print("sucsees in",count)
    return total_1 / count, total_2 / count, total_diff / count,total_multi / count


In [ ]:
#Zeroes the output of the chosen heads at all positions, using hooks that stay attached until reset.
def add_hook_on_heads_ablate(model,heads_to_ablate):

    def ablate_head(z, hook, head):
        z[:, :, head, :] = 0.0  
        return z

    for layer, head in heads_to_ablate:
        model.add_hook(
            utils.get_act_name("z", layer),
            partial(ablate_head, head=head),
            is_permanent=True,
        )

#Same, but zeroes the heads only at the last token position.

def add_hook_on_heads_ablate_last_pos(model,heads_to_ablate):

    def ablate_head(z, hook, head):
        z[:, -1, head, :] = 0.0  
        return z
    

    for layer, head in heads_to_ablate:
        model.add_hook(
            utils.get_act_name("z", layer),
            partial(ablate_head, head=head),
            is_permanent=True,
        )

#Amplifies the chosen heads by multiplying their output
def add_hook_on_heads_amp(model,heads_to_ablate):

    def ablate_head(z, hook, head):
        z[:, :, head, :] = z[:, :, head, :]*6   
        return z
    

    for layer, head in heads_to_ablate:
        model.add_hook(
            utils.get_act_name("z", layer),
            partial(ablate_head, head=head),
            is_permanent=True,
        )

#Amplifies the chosen heads by adding the unembedding vector of each head's own top predicted token to its last-position output.
import torch
from functools import partial
from transformer_lens import utils
model.set_use_attn_result(True)
def add_hook_top_token_unembed_result(model, heads, scale=1.0, verbose=True):
   
    model.set_use_attn_result(True)   

    def add_vec(result, hook, head, layer, scale):
        # result: [batch, pos, head, d_model]
        with torch.no_grad():
            h = result[:, -1:, head, :]                       
            normed = model.ln_final(h)                      
            logits = model.unembed(normed)                  
            top_ids = logits[:, -1].argmax(dim=-1)            

            if verbose:
                toks = [model.to_string(i) for i in top_ids]
                print(f"L{layer}H{head} top token(s): {toks}")

            vec = model.W_U[:, top_ids].T * scale             

        result[:, -1, head, :] += vec                         
        return result

    for layer, head in heads:
        model.add_hook(
            utils.get_act_name("result", layer),
            partial(add_vec, head=head, layer=layer, scale=scale),
            is_permanent=True,
        )

In [ ]:

ds_NQ_Swap = load_dataset("pminervini/NQ-Swap")
ds_NQ_Swap=ds_NQ_Swap["dev"].shuffle(seed=42)
list_of_memory_answers_data_NQ-swap,list_of_context_answers_data_NQ-swap=answer_from_mem_or_context(ds_NQ_Swap)
context_only_train,context_only_test,mem_only_train,mem_only_test=filter_Q_and_split_train_test(list_of_memory_answers_data_NQ,list_of_context_answers_data_NQ)

In [ ]:
total_MEM_MEM_A,total_context_MEM_A,total_diff_MEM_A,total_multi_MEM_A=compute_dataset_importance_diff(mem_only_train,"org_answer","sub_answer",build_conflict_prompt,importance_for_example_prob_sum)

total_context_context_A,total_MEM_context_A,total_diff_context_A,total_multi_context_A=compute_dataset_importance_diff(context_only_train,"sub_answer","org_answer",build_conflict_prompt,importance_for_example_prob_sum)

In [ ]:
'''np.savez(f'/kaggle/working/data_proccesed_mem_metrics_2.npz',
 total1=total_MEM_MEM_A.clone().detach().numpy(),
 total2=total_context_MEM_A.clone().detach().numpy(),
 total_diff=total_diff_MEM_A.clone().detach().numpy(),
 total_multi= total_multi_MEM_A.clone().detach().numpy(),
)
np.savez(f'/kaggle/working/data_proccesed_context_metrics_2.npz',
 total1=total_context_context_A.clone().detach().numpy(),
 total2=total_MEM_context_A.clone().detach().numpy(),
 total_diff=total_diff_context_A.clone().detach().numpy(),
 total_multi= total_multi_context_A.clone().detach().numpy(),
)'''

In [ ]:
import torch
data = np.load('/kaggle/input/datasets/yochaqwami/mem-contetxt-14-9-with-prob-sum/data_proccesed_mem_metrics_2.npz')
total_MEM_MEM_A = torch.from_numpy(data['total1'])
total_context_MEM_A= torch.from_numpy(data['total2'])
total_diff_MEM_A= torch.from_numpy(data['total_diff'])
total_multi_MEM_A= torch.from_numpy(data['total_multi'])

data = np.load('/kaggle/input/datasets/yochaqwami/mem-contetxt-14-9-with-prob-sum/data_proccesed_context_metrics_2.npz')
total_context_context_A = torch.from_numpy(data['total1'])
total_MEM_context_A= torch.from_numpy(data['total2'])
total_diff_context_A= torch.from_numpy(data['total_diff'])
total_multi_context_A= torch.from_numpy(data['total_multi'])

In [ ]:
#Plots several head-importance heatmaps side by side

def plot_heads_grid(matrices, titles, filename, cmap="RdBu"):
    # Set paper typography (Serif font, 8-10pt sizes matching document body)
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 14,
        "axes.titlesize": 20,
        "axes.labelsize": 14,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
    })

    n = len(matrices)
    # 7.0 inches total width fits across two columns (\textwidth)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.6), sharey=True)
    if n == 1:
        axes = [axes]

    for i, (ax, mat, title) in enumerate(zip(axes, matrices, titles)):
        mat_np = to_np(mat)
        vmax = np.abs(mat_np).max()
        im = ax.imshow(mat_np, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
        
        ax.set_title(title, pad=6)
        ax.set_xlabel("Head")
        
        # Keep Y-axis label only on the left-most panel to reduce clutter
        if i == 0:
            ax.set_ylabel("Layer")
        else:
            ax.set_ylabel("")

        # Add scaled colorbar
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Trim whitespace margins
    plt.tight_layout(pad=0.4)
    
    # Export directly to vector PDF
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    plt.show()
    plt.close()



plot_heads_grid(
    [total_MEM_MEM_A, total_context_MEM_A, total_diff_MEM_A],
    ["y_mem Importance", "y_ctx Importance", "Memory Preference Margin"],
    filename="fig_importance_D_mem.pdf",
    cmap="RdBu",
)


plot_heads_grid(
    [total_context_context_A, total_MEM_context_A, total_diff_context_A],
    ["y_ctx Importance", "y_mem Importance", "Context Preference Margin"],
    filename="fig_importance_D_ctx.pdf",
    cmap="RdBu",
)

In [ ]:
fact_NQ_swap=[]
with open('/kaggle/input/datasets/nachmanrog/known-facts-nq-swap/nqswap_style_dataset.json', "r") as f:
    fact_NQ_swap = json.load(f)

list_of_memory_answers_data,list_of_context_answers_data=answer_from_mem_or_context(ds_NQ_Swap)

In [ ]:
file_path_memory_answers = "/kaggle/input/datasets/nachmanrog/fact-nq-mem-context/list_of_memory_answers_data_fact_nq.json"
file_path_context_answers = "/kaggle/input/datasets/nachmanrog/fact-nq-mem-context/list_of_context_answers_data_fact_nq.json"

list_of_memory_answers_data=[]
list_of_context_answers_data=[]
if os.path.exists(file_path_memory_answers):
    with open(file_path_memory_answers, "r") as f:
        list_of_memory_answers_data = json.load(f)


if os.path.exists(file_path_context_answers):
    with open(file_path_context_answers, "r") as f:
        list_of_context_answers_data = json.load(f)

In [ ]:
#Returns the (layer, head) indices of the top-k values in a matrix.

def list_of_k_heads(mat,k):
    flat_values, flat_indices = torch.topk(mat.flatten(), k=k)
    row_indices, col_indices = torch.unravel_index(flat_indices, mat.shape)
    top_k_heads_of_mem = torch.stack([row_indices, col_indices], dim=1)
    return top_k_heads_of_mem


In [ ]:
k=15

greter_z_total_diff_MEM_A=total_diff_MEM_A*(total_diff_MEM_A>0)
flat_values, flat_indices = torch.topk(greter_z_total_diff_MEM_A.flatten(), k=k)
row_indices, col_indices = torch.unravel_index(flat_indices, greter_z_total_diff_MEM_A.shape)
top_k_heads_of_mem = torch.stack([row_indices, col_indices], dim=1)

greter_z_total_diff_context_A=total_diff_context_A*(total_diff_context_A>0)
flat_values, flat_indices = torch.topk(greter_z_total_diff_context_A.flatten(), k=k)
row_indices, col_indices = torch.unravel_index(flat_indices, greter_z_total_diff_context_A.shape)
top_k_heads_of_context = torch.stack([row_indices, col_indices], dim=1)

mem_set = set(map(tuple, top_k_heads_of_mem.tolist()))
context_set = set(map(tuple, top_k_heads_of_context.tolist()))

common_heads = mem_set & context_set

mem_only = mem_set - common_heads
context_only = context_set - common_heads

top_k_heads_of_mem_reduced = torch.tensor(list(mem_only))
top_k_heads_of_context_reduced = torch.tensor(list(context_only))

print(f"Mem-only heads ({len(mem_only)}):", top_k_heads_of_mem_reduced)
print(f"Context-only heads ({len(context_only)}):", top_k_heads_of_context_reduced)

In [ ]:
#Returns unique random (layer, head) pairs, excluding given heads, with a fixed seed.
import random
def random_points(count, n, m, excluded=(), seed=42):
    """Return `count` unique random [x, y] pairs with 0 < x < n and 0 < y < m,
    none of which appear in `excluded`. Same seed -> same result."""
    rng = random.Random(seed)
    banned = {tuple(p) for p in excluded}
    available = (n - 1) * (m - 1) - len(banned)
    if count > available:
        raise ValueError(f"Only {available} valid points available, requested {count}")

    seen = set()
    result = []
    while len(result) < count:
        p = (rng.randint(1, n - 1), rng.randint(1, m - 1))
        if p not in banned and p not in seen:
            seen.add(p)
            result.append(list(p))
    return result

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
#Applies the hook setup to each example, generates an answer, and counts how many came from memory, from context, or from neither.
def run_experiment(name, data_list, setup_hooks_fn, max_new_tokens=50, verbose=True):
    count_from_mem = 0
    count_from_contxt = 0
    none_form_mem_or_context=0
    for data in data_list:
        model.reset_hooks(including_permanent=True)
        setup_hooks_fn()
        batch = [build_conflict_prompt(data)]
        bos_id = model.tokenizer.bos_token_id
        real_input_ids = model.to_tokens(batch, prepend_bos=False)
        batch_size, real_seq_len = real_input_ids.shape
        bos_tensor = torch.full((batch_size, 1), bos_id, device=model.cfg.device, dtype=torch.long)
        input_ids = torch.cat([bos_tensor, real_input_ids], dim=1).to(model.cfg.device)

        pre_output = model.to_string(input_ids)[0]
        with torch.no_grad():
            full_tokens = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False)
        output = model.to_string(full_tokens)[0][len(pre_output):]
        print("output->",output)
        if is_correct(output, data["org_answer"]):
            count_from_mem += 1
        elif is_correct(output, data["sub_answer"]):
            count_from_contxt += 1
        else:
            none_form_mem_or_context+= 1

        if verbose:
            print(f"[{name}] mem={count_from_mem} ctx={count_from_contxt} none={none_form_mem_or_context}")

        model.reset_hooks(including_permanent=True)
        gc.collect()
        torch.cuda.empty_cache()

    n = len(data_list)
    count_other = n - count_from_mem - count_from_contxt
    result = {
        "name": name,
        "n": n,
        "count_mem": count_from_mem,
        "count_ctx": count_from_contxt,
        "count_other": count_other,
        "frac_mem": count_from_mem / n,
        "frac_ctx": count_from_contxt / n,
        "frac_other": count_other / n,
    }
    print(name, "->", result)
    return result

greter_z_total_diff_MEM_A=total_diff_MEM_A*(total_diff_MEM_A>0)

greter_z_total_diff_context_A=total_diff_context_A*(total_diff_context_A>0)


def setup_ablate_mem_15_headse_last_pos():
    add_hook_on_heads_ablate_last_pos(model, list_of_k_heads(greter_z_total_diff_MEM_A,15))

def setup_ablate_mem_amp_ctx_15_headse_last_pos():
    add_hook_on_heads_ablate_last_pos(model, list_of_k_heads(greter_z_total_diff_MEM_A,15))
    add_hook_top_token_unembed_result(model, list_of_k_heads(greter_z_total_diff_context_A,15), scale=5.0, verbose=False)

def setup_random_ablate_from_meme_last_pos():
    add_hook_on_heads_ablate_last_pos(model, random_head_not_from_mem)

def setup_ablate_ctx_15_headse_last_pos():
    add_hook_on_heads_ablate_last_pos(model, list_of_k_heads(greter_z_total_diff_context_A,15))

def setup_ablate_ctx_30_headse_last_pos():
    add_hook_on_heads_ablate_last_pos(model, list_of_k_heads(total_diff_context_A*(total_diff_context_A>0),30))

def setup_ablate_ctx_50_headse_last_pos():
    add_hook_on_heads_ablate_last_pos(model, list_of_k_heads(total_diff_context_A*(total_diff_context_A>0),50))

def setup_ablate_ctx_amp_mem_15_headse_last_pos():
    add_hook_on_heads_ablate_last_pos(model, list_of_k_heads(greter_z_total_diff_context_A,15))
    add_hook_top_token_unembed_result(model, list_of_k_heads(greter_z_total_diff_MEM_A,15), scale=1.0, verbose=False)

def setup_random_ablate_not_ctx():
    add_hook_on_heads_ablate_last_pos(model, random_head_not_from_context)


short_list_of_memory_answers_data=list_of_memory_answers_data[:170]
short_contxt_list_ans = list_of_context_answers_data[:170]
random_head_not_from_mem = random_points(15, model.cfg.n_layers, model.cfg.n_heads, list_of_k_heads(total_diff_MEM_A*(total_diff_MEM_A>0),50))
random_head_not_from_context = random_points(15, model.cfg.n_layers, model.cfg.n_heads, list_of_k_heads(total_diff_context_A*(total_diff_context_A>0),50))

experiments = [
    ("Ablate Mem heads 15_heads",              short_list_of_memory_answers_data, setup_ablate_mem_15_heads),
    ("Ablate Mem + Amplify Ctx 15_heads",      short_list_of_memory_answers_data, setup_ablate_mem_amp_ctx_15_heads),
    ("Random ablation (mem control) 15_heads", short_list_of_memory_answers_data, setup_random_ablate_not_ctx),
    ("Random ablation (ctx control) 15_heads", short_contxt_list_ans,        setup_random_ablate_from_mem),
    ("Ablate Ctx heads 15_heads",              short_contxt_list_ans,        setup_ablate_ctx_15_heads),
    ("Ablate Ctx + Amplify Mem 15_heads",      short_contxt_list_ans,        setup_ablate_ctx_amp_mem_15_heads),
    ("Ablate Ctx heads 30_heads",      short_contxt_list_ans,        setup_ablate_ctx_30_heads),
    ("Ablate Ctx heads 50_heads",      short_contxt_list_ans,        setup_ablate_ctx_50_heads)
]

# run all configs and save results 
results = {}
for name, data_list, setup_fn in experiments:
    results[name] = run_experiment(name, data_list, setup_fn)



In [ ]:
def build_no_conflict_prompt(row,req):

    context = row["sub_context"]
    question = row["question"]
    prompt = f"{context}\n\nQuestion: {question},{req} \nAnswer in few words:"

    return prompt
#Runs the same counting experiment with no head hooks, using the instruction-based prompt (the baseline benchmark).
def run_experimen_no_ablate(name, data_list, req, max_new_tokens=50, verbose=True):
    count_from_mem = 0
    count_from_contxt = 0
    none_form_mem_or_context=0
    for data in data_list:
        model.reset_hooks(including_permanent=True)

        batch = [build_no_conflict_prompt(data,req)]
        bos_id = model.tokenizer.bos_token_id
        real_input_ids = model.to_tokens(batch, prepend_bos=False)
        batch_size, real_seq_len = real_input_ids.shape
        bos_tensor = torch.full((batch_size, 1), bos_id, device=model.cfg.device, dtype=torch.long)
        input_ids = torch.cat([bos_tensor, real_input_ids], dim=1).to(model.cfg.device)

        pre_output = model.to_string(input_ids)[0]
        with torch.no_grad():
            full_tokens = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False)
        output = model.to_string(full_tokens)[0][len(pre_output):]
        print("output->",model.to_string(full_tokens)[0])
        if is_correct(output, data["org_answer"]):
            count_from_mem += 1
        elif is_correct(output, data["sub_answer"]):
            count_from_contxt += 1
        else:
            none_form_mem_or_context+= 1

        if verbose:
            print(f"[{name}] mem={count_from_mem} ctx={count_from_contxt} none={none_form_mem_or_context}")

        model.reset_hooks(including_permanent=True)
        gc.collect()
        torch.cuda.empty_cache()

    n = len(data_list)
    count_other = n - count_from_mem - count_from_contxt
    result = {
        "name": name,
        "n": n,
        "count_mem": count_from_mem,
        "count_ctx": count_from_contxt,
        "count_other": count_other,
        "frac_mem": count_from_mem / n,
        "frac_ctx": count_from_contxt / n,
        "frac_other": count_other / n,
    }
    print(name, "->", result)
    return result




In [ ]:
results["no_ablate_only_prompt_steer_to_mem"] = run_experimen_no_ablate("no_ablate_only_prompt_steer_to_mem",short_contxt_list_ans,"Please answer the question based on your internal memory, ignoring the given context")
results["no_ablate_only_prompt_steer_to_context"] = run_experimen_no_ablate("no_ablate_only_prompt_steer_to_context",short_list_of_memory_answers_data,"Please answer the question based on the given context, ignoring your internal memory")

with open("ablation_results_last_pos.json", "w") as f:
    json.dump(results, f, indent=2)


In [ ]:
import json
results=[]
with open('/kaggle/input/datasets/nanaenana/ablate-tst-170/ablation_results (6).json', 'r') as file:
    results = json.load(file)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np


mem_names = ["Random ablation (mem control) 15_heads","Ablate Mem heads 15_heads", "Ablate Mem + Amplify Ctx 15_heads"]
ctx_names = ["Random ablation (ctx control) 15_heads", "Ablate Ctx heads 15_heads", "Ablate Ctx + Amplify Mem 15_heads"]
ctx_names_15_30_50 = ["Ablate Ctx heads 15_heads", "Ablate Ctx heads 30_heads", "Ablate Ctx heads 50_heads"]
prompt_St = ["no_ablate_only_prompt_steer_to_mem", "no_ablate_only_prompt_steer_to_context"]

display_labels = {
    "Random ablation (mem control) 15_heads":"Random Ablate\n15 heads",
    "Ablate Mem heads 15_heads": "Ablate Memory\n top-15 heads ",
    "Ablate Mem + Amplify Ctx 15_heads": "Ablate Memory\n top-15 heads\n& Amplify Context\ntop-15 heads",
    "Random ablation (ctx control) 15_heads": "Random Ablate\n15 heads",
    "Ablate Ctx heads 15_heads": "Ablate Context\n top-15 heads ",
    "Ablate Ctx + Amplify Mem 15_heads": "Ablate Context\n top-15 heads\n& Amplify Memory\ntop-15 heads",
    "Ablate Ctx heads 30_heads":"Ablate Context\n top-30 heads ",
    "Ablate Ctx heads 50_heads":"Ablate Context\n top-50 heads ",
    "no_ablate_only_prompt_steer_to_mem":"prompt steer\nfrom Context to Memory",
    "no_ablate_only_prompt_steer_to_context":"prompt steer\nfrom Memory to Context",
}

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# 1. INCREASED GLOBAL FONT SIZES
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 20,          # Increased base size (was 11)
    "axes.titlesize": 11,     # Increased title size (was 11)
    "axes.labelsize": 15,     # Increased axis label size (was 11)
    "xtick.labelsize": 13,    # Increased X-tick size (was 9.5)
    "ytick.labelsize": 11,    # Increased Y-tick size (was 9.5)
    "legend.fontsize": 15,   # Increased legend size (was 9.5)
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLOR_MEM, COLOR_CTX, COLOR_OTHER = "#0072B2", "#E69F00", "#009E73"


def plot_group_bar(names, results, filename, display_labels=None,
                    ylabel="% of examples", figwidth=6.4, figheight=3.2):
    
    frac_mem   = [results[n]["frac_mem"] for n in names]
    frac_ctx   = [results[n]["frac_ctx"] for n in names]
    frac_other = [results[n]["frac_other"] for n in names]
    labels = [display_labels.get(n, n) if display_labels else n for n in names]

    x = np.arange(len(names))
    width = 0.26

    fig, ax = plt.subplots(figsize=(figwidth, figheight))
    bars_mem   = ax.bar(x - width, frac_mem,   width, label="Mem",   color=COLOR_MEM,   edgecolor="black", linewidth=0.4)
    bars_ctx   = ax.bar(x,         frac_ctx,   width, label="Ctx",   color=COLOR_CTX,   edgecolor="black", linewidth=0.4)
    bars_other = ax.bar(x + width, frac_other, width, label="Other", color=COLOR_OTHER, edgecolor="black", linewidth=0.4)

    # 2. INCREASED BAR LABEL FONT SIZE
    for bars in (bars_mem, bars_ctx, bars_other):
        ax.bar_label(bars, fmt="%.2f", padding=1.5, fontsize=11.0,fontweight="bold")  # Increased from 7.5

    ax.set_xticks(x)
    
    if display_labels:
        wrapped_labels = ["\n".join(l.split(" + ")) if " + " in l else l for l in labels]
    else:
        wrapped_labels = [l.replace("_", "\n") for l in labels]
        
    ax.set_xticklabels(wrapped_labels, rotation=0, ha="center", rotation_mode="anchor")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, linewidth=0.4, alpha=0.4)
    ax.set_axisbelow(True)

    ax.legend(frameon=False, ncol=3, loc="lower center",
              bbox_to_anchor=(0.5, 1.02), handlelength=1.2, columnspacing=1.0)

    # 3. PREVENT TEXT CLIPPING WITH TIGHT BOUNDS
    plt.tight_layout()
    fig.savefig(filename, bbox_inches="tight")  # Guarantees larger text isn't cut off at edges
    plt.show()




plot_group_bar(
    mem_names, results,
    filename="ablation_bar_memory.pdf",
    display_labels=display_labels,
)

plot_group_bar(
    ctx_names, results,
    filename="ablation_bar_context.pdf",
    display_labels=display_labels,
)

plot_group_bar(
    ctx_names_15_30_50, results,
    filename="ablation_bar_context_15_30_50.pdf",
    display_labels=display_labels,
)